# QVerse — Introduction to Quantum Computing & Programming
        ## Week 11: Grover Search and Amplitude Amplification

        **Level:** Beginner  
        **Recommended study time:** 2–4 hours  
        **Prerequisites:** Weeks 1–10

        ### Learning objectives
        - Explain a marked state and phase oracle.
- Implement a two-qubit Grover search.
- Build a diffuser.
- State Grover's quadratic, not exponential, query speedup.

        ---
        **How to use this notebook**

        1. Read the short theory sections.
        2. Make a prediction before running each guided experiment.
        3. Run and modify the code.
        4. Complete every **TODO** exercise.
        5. Finish the reflection section in your own words.

        The goal is not to memorize syntax. The goal is to connect **quantum idea → circuit → result → explanation**.

In [ ]:
# Run this only if your environment does not have the required packages.
# In a terminal, the preferred setup is:
# python -m pip install "qiskit[visualization]>=2.5" matplotlib numpy

# In a fresh Colab notebook you can instead uncomment:
# %pip install "qiskit[visualization]>=2.5" matplotlib numpy -q

## 1. Search problem

Suppose one bitstring among $N$ possibilities is marked. Classical unstructured search needs $O(N)$ oracle queries in the worst/average asymptotic sense, while Grover's algorithm needs $O(\sqrt N)$ oracle calls.

For two qubits, $N=4$, so a single Grover iteration is enough for the one-target teaching example.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram

def phase_oracle_2q(target):
    if target not in {"00", "01", "10", "11"}:
        raise ValueError("target must be 2 bits")

    qc = QuantumCircuit(2, name=f"O_{target}")

    # Map target to |11>, apply phase flip with CZ, then undo mapping.
    # target uses Qiskit's displayed bitstring convention q1 q0.
    for q, displayed_bit in [(0, target[1]), (1, target[0])]:
        if displayed_bit == "0":
            qc.x(q)

    qc.cz(0, 1)

    for q, displayed_bit in [(0, target[1]), (1, target[0])]:
        if displayed_bit == "0":
            qc.x(q)

    return qc

def diffuser_2q():
    qc = QuantumCircuit(2, name="Diffuser")
    qc.h([0, 1])
    qc.x([0, 1])
    qc.cz(0, 1)
    qc.x([0, 1])
    qc.h([0, 1])
    return qc

## 2. Complete Grover circuit

In [ ]:
def grover_2q(target):
    qc = QuantumCircuit(2)
    qc.h([0, 1])
    qc.append(phase_oracle_2q(target).to_gate(), [0, 1])
    qc.append(diffuser_2q().to_gate(), [0, 1])
    qc.measure_all()
    return qc

for target in ["00", "01", "10", "11"]:
    qc = grover_2q(target)
    counts = StatevectorSampler(seed=4).run([qc], shots=500).result()[0].data.meas.get_counts()
    print(target, counts)

## 3. Compare zero and one Grover iterations

In [ ]:
target = "10"

uniform = QuantumCircuit(2)
uniform.h([0, 1])
uniform.measure_all()

grover = grover_2q(target)

sampler = StatevectorSampler(seed=23)
counts_uniform = sampler.run([uniform], shots=1000).result()[0].data.meas.get_counts()
counts_grover = sampler.run([grover], shots=1000).result()[0].data.meas.get_counts()

print("No Grover iteration:", counts_uniform)
print("One Grover iteration:", counts_grover)
plot_histogram([counts_uniform, counts_grover], legend=["uniform", "Grover"])

## 4. Remove the diffuser

In [ ]:
qc_no_diffuser = QuantumCircuit(2)
qc_no_diffuser.h([0,1])
qc_no_diffuser.append(phase_oracle_2q("10").to_gate(), [0,1])
qc_no_diffuser.measure_all()

print(StatevectorSampler(seed=3).run([qc_no_diffuser], shots=1000).result()[0].data.meas.get_counts())

## Core exercises
1. Implement and test targets `00`, `01`, `10`, and `11`.
2. Compare the success rate with zero versus one Grover iteration.
3. Remove the diffuser and explain why the oracle phase alone is not visible in Z-basis counts.
4. Explain why Grover provides a quadratic rather than exponential speedup.

In [ ]:
# TODO: Write your solutions here.
# Add extra code cells when useful.

## Optional stretch challenge
Extend Grover search to three qubits for one target. Test 0, 1, 2, and 3 Grover iterations and plot success probability.

In [ ]:
# OPTIONAL TODO: Attempt the stretch challenge here.

## Weekly reflection
- What does the oracle change?
- What does the diffuser accomplish intuitively?
- Why can too many Grover iterations reduce the success probability?

## Submission checklist
- [ ] I made at least one prediction before executing a circuit.
- [ ] All guided examples run.
- [ ] I completed the core exercises.
- [ ] I explained the important output rather than only displaying it.
- [ ] My notebook is readable from top to bottom.